In [12]:
# Program for exploring the recording database: Polish spoken words dataset (PSD).
# Reproduces the sounds of words and presents them on charts.
# It is possible to reproduce the sounds of entire sentences or individual phonemes.

directory_name = "../data/raw/"

# directory_name += "1-500"
directory_name += "501-1000"
# directory_name += "1001-1500"
# directory_name += "1501-2000"
# directory_name += "2001-2500"
# directory_name += "2501-3000"
# directory_name += "124"

if_chart_in_the_time_domain = True
if_spectrogram = True

import numpy as np
import matplotlib.pyplot as plt
from os.path import dirname, join as pjoin
from scipy.io import wavfile
import scipy.io
import pdb
import sounddevice as sd
import os
import torch
import torchaudio
import torchaudio.transforms as T

In [13]:
N_FFT = 512  # ile próbek w oknie FFT, czyli rozdzielczość częstotliwościowa spektrogramu
N_MELS = 80  # ile pasm częstotliwościowych w spektrogramie melowym
# im mniejsze to jest tym większa pikseloza,
HOP_LENGTH = N_FFT // 2  # 256 — default used by torchaudio transforms
fs = 16  # czcionka do wykresów
WORDS_TO_SHOW = 19  # slowwa do pokazania na wykresie

rownanie mela: (tu trzeba walnac loga bo decybele i slyszenie ludzi sa logarytmiczne)
$$m = 2595 \cdot \log_{10}\left(1 + \frac{f}{700}\right)$$

In [14]:
def parse_phonemes(text_grid):
    """
    Parse the 'phones' IntervalTier from a TextGrid string.
    Returns a list of (xmin, xmax, text) tuples.
    """
    phonemes = []
    lines = text_grid.split('\n')

    # Find the phones tier
    in_phones_tier = False
    in_interval    = False
    xmin = xmax = None

    for line in lines:
        line = line.strip()

        # Detect entry into the phones tier
        if 'name = "phones"' in line:
            in_phones_tier = True
            continue

        # Detect entry into the next tier (stop reading)
        if in_phones_tier and line.startswith('name =') and 'phones' not in line:
            break

        if not in_phones_tier:
            continue

        # Parse interval fields
        if line.startswith('xmin =') and 'intervals' not in line:
            xmin = float(line.split('=')[1].strip())
        elif line.startswith('xmax =') and 'intervals' not in line:
            xmax = float(line.split('=')[1].strip())
        elif line.startswith('text ='):
            text = line.split('=')[1].strip().strip('"')
            if xmin is not None and xmax is not None:
                phonemes.append((xmin, xmax, text))
            xmin = xmax = None

    return phonemes


def get_word_phonemes(phonemes, time_start, time_end):
    """
    Return phonemes that fall within [time_start, time_end],
    with times converted to be relative to time_start.
    """
    result = []
    for (xmin, xmax, text) in phonemes:
        if xmin >= time_start - 1e-6 and xmax <= time_end + 1e-6:
            result.append((xmin - time_start, xmax - time_start, text))
    return result


def seconds_to_frame(t, sample_rate, hop_length):
    return t * sample_rate / hop_length

In [15]:
done = False

author_directories = [f for f in os.listdir(directory_name) if os.path.isdir(directory_name+"/"+f)]
for aut_dir in author_directories:
    aut_path = directory_name + "/" + aut_dir
    files = [f for f in os.listdir(aut_path) if os.path.isfile(aut_path + "/" + f)]
    sentence_names = []
    for file in files:
        name_parts = file.split(".")
        if name_parts[-1] == "txt": sentence_names.append(name_parts[0])

    for sentence_name in sentence_names:
        wav_fname        = aut_path + "/" + sentence_name + ".wav"
        text_file_name   = aut_path + "/" + sentence_name + ".txt"
        TextGrid_file_name = aut_path + "/" + sentence_name + ".TextGrid"

        samplerate, data = wavfile.read(wav_fname)
        num_of_samples   = data.shape[0]
        length           = num_of_samples / samplerate
        print(f"length = {length}s, samplerate = {samplerate}1/s")

        text_file_ptr = open(text_file_name, 'r', encoding="utf-8")
        text = text_file_ptr.read()
        text_file_ptr.close()

        TextGrid_file_ptr = open(TextGrid_file_name, 'r', encoding="utf-8")
        text_grid = TextGrid_file_ptr.read()
        TextGrid_file_ptr.close()

        # Parse all phonemes from this TextGrid
        all_phonemes = parse_phonemes(text_grid)

        text = text.lower()
        words = text.split(' ')
        grid_lines = text_grid.split('\n')
        grid_line_index = 0
        cleaned_words = []
        for word in words:
            while word[-1] in punctuation_marks:
                word = word[:-1]
                if len(word) == 0: break
            if len(word) > 0: cleaned_words.append(word)

        ct = 0
        for word in cleaned_words:
            print(word)
            for i in range(len(grid_lines)-grid_line_index):
                tokens = grid_lines[i+grid_line_index].split(' ')
                if len(tokens) >= 3:
                    if (tokens[1] == "=") & (tokens[2][1:-1] == word):
                        time_start = float(grid_lines[i+grid_line_index-2].split(' ')[2])
                        time_end   = float(grid_lines[i+grid_line_index-1].split(' ')[2])
                        print("  time: start = " + str(time_start) + ", end = " + str(time_end))
                        sample_no_start = int(time_start / length * num_of_samples)
                        sample_no_end   = int(time_end   / length * num_of_samples)
                        word_data = data[sample_no_start:sample_no_end]

                        sd.play(word_data, samplerate=samplerate)

                        waveform = torch.tensor(word_data, dtype=torch.float32).unsqueeze(0)

                        spec_db     = T.AmplitudeToDB()(T.Spectrogram(
                                          n_fft=N_FFT, hop_length=HOP_LENGTH)(waveform))
                        mel_spec_db = T.AmplitudeToDB()(T.MelSpectrogram(
                                          sample_rate=samplerate,
                                          n_fft=N_FFT,
                                          hop_length=HOP_LENGTH,
                                          n_mels=N_MELS)(waveform))

                        # Get phonemes for this word, relative to word start
                        word_phonemes = get_word_phonemes(all_phonemes, time_start, time_end)

                        # ── One figure per word, 1 row x 3 cols ──────────────
                        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
                        fig.suptitle(f'Word: "{word}"', fontsize=14)

                        # Col 0 — Amplitude vs Time
                        time_axis = np.linspace(0, len(word_data) / samplerate, len(word_data))
                        axes[0].plot(time_axis, word_data)
                        axes[0].set_title("Amplitude vs Time")
                        axes[0].set_xlabel("Time (s)")
                        axes[0].set_ylabel("Amplitude")
                        for (pmin, pmax, ptext) in word_phonemes:
                            axes[0].axvline(x=pmin, color='red', linewidth=0.8, alpha=0.7)
                            axes[0].text(pmin, axes[0].get_ylim()[1], ptext,
                                         fontsize=fs, color='red', rotation=0, va='top')

                        # Col 1 — Normal spectrogram
                        spec_np = spec_db[0].numpy()
                        im1 = axes[1].imshow(spec_np, aspect='auto', origin='lower')
                        axes[1].set_title("Spectrogram")
                        axes[1].set_xlabel("Time frame")
                        axes[1].set_ylabel("Frequency bin")
                        plt.colorbar(im1, ax=axes[1])
                        for (pmin, pmax, ptext) in word_phonemes:
                            frame = seconds_to_frame(pmin, samplerate, HOP_LENGTH)
                            axes[1].axvline(x=frame, color='red', linewidth=0.8, alpha=0.7)
                            axes[1].text(frame, spec_np.shape[0], ptext,
                                         fontsize=fs, color='red', rotation=0, va='top')

                        # Col 2 — Mel spectrogram
                        mel_np = mel_spec_db[0].numpy()
                        im2 = axes[2].imshow(mel_np, aspect='auto', origin='lower')
                        axes[2].set_title("Mel Spectrogram")
                        axes[2].set_xlabel("Time frame")
                        axes[2].set_ylabel("Mel bin")
                        plt.colorbar(im2, ax=axes[2])
                        for (pmin, pmax, ptext) in word_phonemes:
                            frame = seconds_to_frame(pmin, samplerate, HOP_LENGTH)
                            axes[2].axvline(x=frame, color='red', linewidth=0.8, alpha=0.7)
                            axes[2].text(frame, mel_np.shape[0], ptext,
                                         fontsize=fs, color='red', rotation=0, va='top')

                        plt.tight_layout()
                        plt.show()

                        grid_line_index += i + 1
                        break

            ct += 1
            print("sentence " + str(ct) + " finished")
            if ct >= WORDS_TO_SHOW:
                done = True
                break

        if done: break
    if done: break

length = 0.7080625s, samplerate = 160001/s


NameError: name 'punctuation_marks' is not defined